<a href="https://colab.research.google.com/github/gravityeffect1/ARROW/blob/main/ARROW.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ARROW — Automated RNA Recognition and Optimization Workflow
CRISPR-Cas13 guide RNA design for SARS-CoV-2 via NCBI GenBank

**Author:** Sara Șova · 2nd Year Medicine, UMFCD
**ORCID:** [0009-0004-2425-116X](https://orcid.org/0009-0004-2425-116X) · [github.com/gravityeffect1](https://github.com/gravityeffect1)

For research use only. Experimental validation required before any application.


In [ ]:
%pip install biopython -q

import math
import random
import getpass
import warnings

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from Bio import Entrez, SeqIO

warnings.filterwarnings("ignore")



In [ ]:
NCBI_EMAIL = getpass.getpass("NCBI email: ").strip()
if not NCBI_EMAIL or "@" not in NCBI_EMAIL or "." not in NCBI_EMAIL.split("@")[-1]:
    raise ValueError(f"'{NCBI_EMAIL}' is not a valid email address.")

ACCESSION    = "NC_045512.2"
REGION_START = 21563   # Spike (S) protein
REGION_END   = 25384

# Other regions (uncomment to use):
# REGION_START, REGION_END = 28274, 29533  # Nucleocapsid (N)
# REGION_START, REGION_END = 25393, 26220  # Envelope (E)
# REGION_START, REGION_END = 26523, 27191  # Membrane (M)
# REGION_START, REGION_END =   266, 13468  # ORF1a
# REGION_START, REGION_END = 13468, 21555  # ORF1b

GUIDE_LENGTH = 28    #recommended 22-30
GC_MIN       = 0.30
GC_MAX       = 0.70
HP_MAX_RUN   = 4
N_GUIDES     = 50
RANDOM_SEED  = None

KNOWN_REGIONS = {
    (21563, 25384): "Spike (S)",
    (28274, 29533): "Nucleocapsid (N)",
    (25393, 26220): "Envelope (E)",
    (26523, 27191): "Membrane (M)",
    (  266, 13468): "ORF1a",
    (13468, 21555): "ORF1b",
}
REGION_LABEL = KNOWN_REGIONS.get((REGION_START, REGION_END), f"{REGION_START}-{REGION_END}")


In [ ]:
Entrez.email = NCBI_EMAIL
handle = Entrez.efetch(
    db="nucleotide",
    id=ACCESSION,
    rettype="fasta",
    retmode="text",
    seq_start=REGION_START,
    seq_stop=REGION_END,
)
record = SeqIO.read(handle, "fasta")
handle.close()

RNA_SEQ     = str(record.seq).upper().replace("T", "U")
GENOME_NAME = record.description.split(",")[0].strip()

print(GENOME_NAME)
print(f"{len(RNA_SEQ):,} nt fetched  |  GC {sum(b in 'GC' for b in RNA_SEQ)/len(RNA_SEQ)*100:.1f}%")
print(RNA_SEQ[:80] + "...")


In [ ]:
def gc(seq):
    return sum(b in "GC" for b in seq) / len(seq)

def has_homopolymer(seq, n):
    return any(b * n in seq for b in "ACGU")

candidates = []
for i in range(len(RNA_SEQ) - GUIDE_LENGTH + 1):
    guide = RNA_SEQ[i:i + GUIDE_LENGTH]
    g = gc(guide)
    if g < GC_MIN or g > GC_MAX:
        continue
    if has_homopolymer(guide, HP_MAX_RUN):
        continue
    candidates.append({
        "position":   REGION_START + i,
        "sequence":   guide,
        "gc_pct":     round(g * 100, 1),
        "gc_frac":    round(g, 4),
        "count_A":    guide.count("A"),
        "count_U":    guide.count("U"),
        "count_G":    guide.count("G"),
        "count_C":    guide.count("C"),
        "approx_aa":  math.floor(i / 3) + 1,
        "frame":      (REGION_START + i) % 3,
    })

df_all = pd.DataFrame(candidates)
df = df_all.sample(n=min(N_GUIDES, len(df_all)), random_state=RANDOM_SEED).reset_index(drop=True)

print(f"{len(df_all):,} candidates passed filters  →  {len(df)} returned")
print(f"GC: {df.gc_pct.min():.1f}–{df.gc_pct.max():.1f}%  (mean {df.gc_pct.mean():.1f}%)")


In [ ]:
df.sort_values("position").reset_index(drop=True)


In [ ]:
out = df.sort_values("position").reset_index(drop=True)
out.index += 1
out.to_csv("ARROW_gRNA_candidates.csv")

try:
    from google.colab import files
    files.download("ARROW_gRNA_candidates.csv")
except ImportError:
    pass

# top 10 nearest to 50% GC
top10 = df.assign(dev=lambda d: (d.gc_pct - 50).abs()).sort_values("dev").head(10)
print(f"top 10 guides by proximity to 50% GC  ({REGION_LABEL})")
print()
for _, row in top10.iterrows():
    print(f"  {int(row.position):>6,}  {row.sequence}  {row.gc_pct:.1f}%  aa~{int(row.approx_aa)}")


In [ ]:
# change this to inspect any guide (1-based, sorted by position)
GUIDE_INDEX = 1

row = df.sort_values("position").reset_index(drop=True).iloc[GUIDE_INDEX - 1]

print(f"position   {int(row.position):,} nt")
print(f"sequence   {row.sequence}")
print(f"GC         {row.gc_pct:.1f}%  ({'optimal' if 40 <= row.gc_pct <= 60 else 'low' if row.gc_pct < 40 else 'high'})")
print(f"bases      A={int(row.count_A)}  U={int(row.count_U)}  G={int(row.count_G)}  C={int(row.count_C)}")
print(f"approx AA  {int(row.approx_aa)}")
print(f"frame      {int(row.frame)}")
print(f"homopolymer  pass (no run >= {HP_MAX_RUN})")
